# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/chaitanyamadire01-dotcom/flyrank/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

Task Type: Binary Classification & Ranking / Scoring

We formulate the task as predicting the probability that a published URL will experience sustained performance decay (P(is_declining = 1)), and then ranking those pages by an actionable opportunity score. While clustering could group similar pages, our goal is decision prioritization: sorting URLs by expected traffic loss so content teams know exactly which pages to inspect and refresh first.

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

Target / Proxy: trend_direction == 'down' (Binary Flag: 1 = decaying, 0 = stable/growing)

Source: This is an observed outcome computed over a 90-day historical window from aggregated search impressions and user clicks (impressions_90d, clicks_90d, avg_position), rather than an arbitrary subjective label.

Proxy Nature: While the ultimate business goal is "lost revenue or lost user acquisition," trend_direction == 'down' serves as an observable proxy indicating persistent organic visibility loss before a page falls off Page 1 completely.

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Success metric

*One metric you can defend. What number means 'good'?*

Primary Evaluation Metric: Precision@K (specifically Precision@50)Defense: An editorial content team has finite bandwidth and typically refreshes between 20 and 50 articles per sprint. Broad metrics like global ROC-AUC or overall Accuracy are misleading because more than half the catalog is decaying (54.2% base rate). If our top 50 recommendations contain stable pages (false positives), human editor hours (~3 hours/article) are wasted rewriting content that didn't need intervention.What number means "good": A Precision@50 $\ge$ 0.70 (at least 35 of the top 50 flagged pages are genuine decaying opportunities) significantly outperforms the ~24% to 54% heuristic baselines and provides clear operational value.

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

In [10]:
from pathlib import Path
import numpy as np
import pandas as pd

# 1. Locate and load the dataset reliably
candidate_paths = [
    Path("data/raw/content_refresh_anonymized.csv"),
    Path("../../data/raw/content_refresh_anonymized.csv"),
    Path("../data/raw/content_refresh_anonymized.csv"),
]
data_path = next((p for p in candidate_paths if p.exists()), None)
if data_path is None:
    raise FileNotFoundError("Could not find content_refresh_anonymized.csv.")

df_raw = pd.read_csv(data_path)

# 2. Define the Target Column: Binary decay indicator
df_lane = df_raw.copy()
df_lane['target_is_declining'] = (df_lane['trend_direction'].str.lower() == 'down').astype(int)

# 3. Add an Opportunity Proxy: estimated lost traffic potential
df_lane['opportunity_volume'] = df_lane['impressions_90d'] * df_lane['target_is_declining']

# 4. Target columns to display (dynamically keeping only available ones)
candidate_cols = [
    'client_id', 'url', 'page_id', 'page_path', 'avg_position', 
    'impressions_90d', 'clicks_90d', 'ctr', 'position_tier', 
    'content_age_days', 'target_is_declining', 'opportunity_volume'
]
display_cols = [c for c in candidate_cols if c in df_lane.columns]

# Show unit of analysis: 1 row = 1 unique client page over a 90-day window
print(f"Total rows (pages): {len(df_lane):,}")
print("Unit of analysis: 1 row = 1 published URL/page for a given client over a 90-day rolling window.\n")
print("Target distribution:")
print(df_lane['target_is_declining'].value_counts(normalize=True).rename({1: 'Decaying (1)', 0: 'Stable/Up (0)'}))

# Preview top 5 rows of formatted slice
df_lane[display_cols].head()

Total rows (pages): 30,000
Unit of analysis: 1 row = 1 published URL/page for a given client over a 90-day rolling window.

Target distribution:
target_is_declining
Decaying (1)     0.542067
Stable/Up (0)    0.457933
Name: proportion, dtype: float64


,client_id,avg_position,impressions_90d,clicks_90d,ctr,position_tier,content_age_days,target_is_declining,opportunity_volume
0,client_f369cb89fc,10.6,3803,29,0.76,striking,187,1,3803
1,client_4e07408562,20.3,15320,7,0.05,page_3_5,445,1,15320
2,client_7f2253d7e2,36.5,12581,11,0.09,page_3_5,141,1,12581
3,client_19581e27de,6.2,11751,58,0.49,page_1,463,0,0
4,client_3fdba35f04,44.0,19140,24,0.13,page_3_5,263,1,19140


In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

Why ML Beats Fixed Heuristics:Non-linear interactions: A simple rule like "refresh any page with avg_position > 10" fails because pages in positions 4–10 ("striking distance") experience sharp CTR cliffs and high recovery potential, while low-impression pages beyond position 20 have negligible traffic yield.Decoupled features: Search volume has virtually no correlation with impressions (Pearson $r = 0.001$). A fixed rule based on high keyword search volume wastes time on empty queries.Empirical evidence from starter notebooks: A simple hand-written rule achieved only ~24% precision on discovering decay, whereas a shallow decision tree captures non-linear splits across impressions, CTR, and age to triple discovery precision in the top tiers.

In [12]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

- [x] Named the ML task type (Binary Classification & Ranking/Scoring).
- [x] Defined target/proxy (`target_is_declining` from 90-day `trend_direction`).
- [x] Defended success metric (Precision@50 for finite sprint bandwidth).
- [x] Displayed unit of analysis as a real dataframe (1 row = 1 URL/page over 90 days).
- [x] Explained why ML beats fixed heuristics with real data arguments.